In [1]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

In [7]:
def print_structure(h5file: h5py.File) -> None:
    print(f"Contents of {h5file.filename}:")

    def visitor(name: str, obj: h5py.Dataset | h5py.Group) -> None:
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape} dtype={obj.dtype}")

    h5file.visititems(visitor)

In [24]:
path = "/nfs-share/pa511/code_bases/new_jac/results/signs.h5"
with h5py.File(path, "r") as f:
    print_structure(f)

    dataset_name = None
    if dataset_name is None:
        dataset_name = next(iter(f.keys()))
        print(f"\nNo --dataset given, defaulting to '{dataset_name}'")

    data = np.asarray(f[dataset_name])

print(f"\nLoaded '{dataset_name}' with shape {data.shape}, dtype {data.dtype}")

Contents of /nfs-share/pa511/code_bases/new_jac/results/signs.h5:
  signs: shape=(32, 2, 8) dtype=float32

No --dataset given, defaulting to 'signs'

Loaded 'signs' with shape (32, 2, 8), dtype float32


In [25]:
data.shape

(32, 2, 8)

In [26]:
data[0][0][0]

np.float32(-1.0)

In [21]:
path = "/nfs-share/pa511/code_bases/new_jac/results/singular_values.h5"
with h5py.File(path, "r") as f:
    print_structure(f)

    dataset_name = None
    if dataset_name is None:
        dataset_name = next(iter(f.keys()))
        print(f"\nNo --dataset given, defaulting to '{dataset_name}'")

    sing_vals = np.asarray(f[dataset_name])

print(f"\nLoaded '{dataset_name}' with shape {data.shape}, dtype {data.dtype}")

Contents of /nfs-share/pa511/code_bases/new_jac/results/singular_values.h5:
  singular_values: shape=(32, 2, 8, 896) dtype=float32

No --dataset given, defaulting to 'singular_values'

Loaded 'singular_values' with shape (32, 2, 8, 896), dtype float32


In [28]:
path = "/nfs-share/pa511/code_bases/new_jac/results/log_determinants.h5"
with h5py.File(path, "r") as f:
    print_structure(f)

    dataset_name = None
    if dataset_name is None:
        dataset_name = next(iter(f.keys()))
        print(f"\nNo --dataset given, defaulting to '{dataset_name}'")

    dets = np.asarray(f[dataset_name])

print(f"\nLoaded '{dataset_name}' with shape {data.shape}, dtype {data.dtype}")

Contents of /nfs-share/pa511/code_bases/new_jac/results/log_determinants.h5:
  log_determinants: shape=(32, 2, 8) dtype=float32

No --dataset given, defaulting to 'log_determinants'

Loaded 'log_determinants' with shape (32, 2, 8), dtype float32


In [39]:
sing_vals[0][1][0]

array([7.48415146e+01, 3.97363129e+01, 3.59526749e+01, 3.18583412e+01,
       2.91552372e+01, 2.70312424e+01, 2.34861736e+01, 2.20325871e+01,
       2.05742188e+01, 1.99557648e+01, 1.93356915e+01, 1.90769577e+01,
       1.86236210e+01, 1.77452717e+01, 1.76558361e+01, 1.71947708e+01,
       1.70417233e+01, 1.69541798e+01, 1.67717781e+01, 1.63669662e+01,
       1.62789726e+01, 1.61575871e+01, 1.56458559e+01, 1.53891258e+01,
       1.52604103e+01, 1.51947680e+01, 1.49725752e+01, 1.48718758e+01,
       1.47680120e+01, 1.45310726e+01, 1.43444042e+01, 1.43178310e+01,
       1.41673250e+01, 1.39630976e+01, 1.38818245e+01, 1.37589970e+01,
       1.37242260e+01, 1.34763260e+01, 1.33958359e+01, 1.32722492e+01,
       1.31095772e+01, 1.30249996e+01, 1.29677896e+01, 1.29011393e+01,
       1.27723207e+01, 1.26913080e+01, 1.25795250e+01, 1.24510860e+01,
       1.24375753e+01, 1.24042120e+01, 1.22030201e+01, 1.21737680e+01,
       1.20948477e+01, 1.20229998e+01, 1.18473368e+01, 1.18239393e+01,
      

In [43]:
dets[0][1][0]

np.float32(950.7824)

In [41]:
import numpy as np
from scipy.stats import ortho_group

def construct_dense_matrix(sigma: np.ndarray, J: float, tol: float = 1e-7) -> np.ndarray:
    """
    Constructs a dense square matrix A with given singular values and determinant J.
    
    Parameters:
        sigma (np.ndarray): 1D array of singular values (must be non-negative).
        J (float): Desired determinant (Jacobian determinant).
        tol (float): Relative tolerance to verify |J| == prod(sigma).
        
    Returns:
        np.ndarray: Dense n x n matrix A.
    """
    sigma = np.asarray(sigma, dtype=np.float64)
    if sigma.ndim != 1:
        raise ValueError("sigma must be a 1D array.")
    if np.any(sigma < 0):
        raise ValueError("Singular values must be non-negative.")
    
    n = len(sigma)
    prod_sigma = np.prod(sigma)
    
    # Check if |J| equals prod(sigma) within tolerance
    if not np.isclose(np.abs(J), prod_sigma, rtol=tol, atol=tol):
        raise ValueError(
            f"Incompatible determinant: |J| ({np.abs(J)}) must equal prod(sigma) ({prod_sigma})."
        )
    
    target_sign = np.sign(J)
    
    # Generate random Haar-distributed orthogonal matrices
    U = ortho_group.rvs(dim=n)
    V = ortho_group.rvs(dim=n)
    
    # Adjust sign of det(U) * det(V) to match sgn(J)
    det_product = np.linalg.det(U) * np.linalg.det(V)
    if np.sign(det_product) != target_sign:
        U[:, 0] *= -1  # Flip one column to invert determinant sign
        
    # Construct A = U @ diag(sigma) @ V.T
    A = U @ np.diag(sigma) @ V.T
    return A

In [46]:
A = construct_dense_matrix(sing_vals[0][1][0],np.pow(10, dets[0][1][0]))

/tmp/ipykernel_910336/1472284109.py:1: RuntimeWarning: overflow encountered in power
  A = construct_dense_matrix(sing_vals[0][1][0],np.pow(10, dets[0][1][0]))


In [45]:
np.linalg.slogdet(A)

SlogdetResult(sign=np.float64(1.0), logabsdet=np.float64(950.9081408753066))

In [49]:
ans = np.linalg.svdvals(A)

In [50]:
ans-sing_vals[0][1][0]

array([ 2.84217094e-14, -1.42108547e-14, -7.10542736e-15, -3.55271368e-15,
        1.77635684e-14, -1.42108547e-14,  3.55271368e-15,  0.00000000e+00,
        7.10542736e-15,  3.55271368e-15,  2.13162821e-14,  1.42108547e-14,
       -2.13162821e-14, -7.10542736e-15, -3.90798505e-14, -1.06581410e-14,
        3.19744231e-14, -1.42108547e-14,  7.10542736e-15,  2.48689958e-14,
       -2.13162821e-14,  2.13162821e-14, -1.77635684e-14,  2.48689958e-14,
        0.00000000e+00, -3.01980663e-14,  3.01980663e-14, -3.01980663e-14,
        1.77635684e-15,  3.90798505e-14,  1.24344979e-14,  1.77635684e-14,
       -2.30926389e-14,  1.06581410e-14,  1.24344979e-14,  3.55271368e-14,
        3.01980663e-14,  1.42108547e-14, -7.10542736e-15, -1.06581410e-14,
        3.01980663e-14, -3.55271368e-15,  0.00000000e+00,  1.95399252e-14,
        2.66453526e-14, -4.61852778e-14,  1.42108547e-14,  4.44089210e-14,
        3.37507799e-14,  2.66453526e-14,  2.66453526e-14, -5.32907052e-15,
        4.26325641e-14,  

In [52]:
path = "/nfs-share/pa511/code_bases/new_jac/results/eigen_vals.h5"
with h5py.File(path, "r") as f:
    print_structure(f)

    dataset_name = None
    if dataset_name is None:
        dataset_name = next(iter(f.keys()))
        print(f"\nNo --dataset given, defaulting to '{dataset_name}'")

    eigen = np.asarray(f[dataset_name])

print(f"\nLoaded '{dataset_name}' with shape {data.shape}, dtype {data.dtype}")

Contents of /nfs-share/pa511/code_bases/new_jac/results/eigen_vals.h5:
  singular_values: shape=(32, 2, 8, 896) dtype=complex64

No --dataset given, defaulting to 'singular_values'

Loaded 'singular_values' with shape (32, 2, 8), dtype float32


In [ ]:
eigen[0][0][0]